# 03 — Stage 8: generation, and judging the judges

Retrieval is held **byte-identical** across every candidate — same parser, chunking,
embeddings, store and fusion — so any difference in the generation metrics is
attributable to the generator and not to what it was given to read.

Generation and judging are separate scripts on purpose. Generation is the expensive,
irreplaceable step, so its output is cached to disk; a judge framework falling over
then costs a re-score, not a re-generation. That decision paid for itself repeatedly.

In [ ]:
import sys, json; sys.path.insert(0, "../src"); sys.path.insert(0, "../scripts")
import pandas as pd
from pathlib import Path

RESULTS = Path("../reports/results")

rows = []
for path in sorted(RESULTS.glob("generations__*.json")):
    payload = json.loads(path.read_text())
    recs = payload["records"]
    live = [r for r in recs if not r["blocked"]]
    rows.append({
        "model": payload["model"],
        "answers": len(recs),
        "route acc": sum(r["predicted_route"] == r["gold_route"] for r in recs) / len(recs),
        "valid citations": sum(r["citations_valid"] for r in live) / max(len(live), 1),
        "mean gen ms": sum(r["generation_ms"] for r in live) / max(len(live), 1),
        "mean answer chars": sum(len(r["answer"]) for r in live) / max(len(live), 1),
    })
pd.DataFrame(rows).round(3)

These four columns are **quota-free** — they fall out of the generations themselves and
need no LLM judge. `route acc` is identical across models by construction (routing
happens before generation), which is a useful sanity check that retrieval really was
held fixed. `valid citations` is the interesting one: it measures whether the model
actually cited sources that exist, and it is a cheap, objective proxy for groundedness
that no judge can dispute.

## The judged metrics

In [ ]:
pd.DataFrame(json.loads((RESULTS / "stage8_generation.json").read_text()))

## Why the two judges are different models

The methodology treats RAGAS and DeepEval as two independent second opinions. That only
means something if the underlying judges actually differ — running both on the same
model measures one model's opinion twice and then calls the agreement corroboration.

So RAGAS is judged by `gemini-3.6-flash` and DeepEval by `gemini-3.5-flash-lite`:
different generation *and* different tier. This is **cross-generation, not
cross-vendor**, which is weaker independence than two different providers would give —
stated plainly rather than glossed.

## Getting the judges to work at all

Four separate obstacles, each of which silently produced `NaN` rather than an error:

| Problem | Cause | Fix |
|---|---|---|
| RAGAS `ModuleNotFoundError` | RAGAS imports `langchain_community.chat_models.vertexai`, which current langchain-community no longer ships | stub module shim, registered before the RAGAS import |
| DeepEval rejected the model | `GPTModel` validates names against a hard-coded OpenAI whitelist | native `GeminiModel` / `OllamaModel` |
| Judging took ~175 s/record locally | Ollama allocates each model's *maximum* context — 131,072 tokens for Llama 3.1, reserving ~22 GB | judge models rebuilt with `num_ctx 8192` |
| Half of all metrics returned `NaN` | Gemini free tier allows ~20 requests/min **per model**; 8 concurrent workers blew through it, and RAGAS's own 180 s job timeout also expired | in-process token bucket at 12 RPM, `thinking_budget=0`, `RunConfig(timeout=1800)` |

The last row is the one worth remembering: **a rate-limited judge does not raise, it
returns `NaN`**, and a mean over mostly-`NaN` values looks like a real score. Any
evaluation harness that silently coerces failures into numbers will report confident
nonsense, which is why the loader below counts what was actually scored.

In [ ]:
from stage8_evaluate import load_records

for model in ("llama3.2:3b", "qwen2.5:7b", "llama3.1:latest"):
    try:
        _, stats = load_records(model)
        print(f"{model:<18} scored={stats['n_scored']:<3} blocked={stats['n_blocked']} "
              f"route_acc={stats['route_accuracy']:.3f} citations={stats['citations_valid']:.3f}")
    except FileNotFoundError:
        print(f"{model:<18} (no generations on disk)")

Guardrail-blocked answers are excluded from the generation metrics: they have no
retrieval context to be faithful to, so scoring them would penalise a model for the
guardrail working correctly.

## The custom G-Eval criterion

Generic answer-relevancy would pass a fluent answer that quietly drops the `$1,075`
denied-boarding cap or the 60-day cargo claim deadline. For a bot answering questions
about money and deadlines, that is the failure that matters, so the DeepEval G-Eval
criterion rewards exact figures and penalises both invented fees and omitted
conditions — see `PolicyAccuracy` in `src/delta_rag/evaluation.py`.